# 02 — Baseline Models (SVM & Random Forest)

We use **MFCC statistical features** (mean + std over time = 80-dimensional vector)
as input to classical ML classifiers.

**Prerequisite:** run `python extract_representations/extract_features.py` first.

In [ ]:
import sys; sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
import warnings; warnings.filterwarnings('ignore')

EMOTION_NAMES = ['neutral','calm','happy','sad','angry','fearful']
plt.rcParams['figure.dpi'] = 120
sns.set_style('whitegrid')

## 1. Load Pre-computed Features

In [ ]:
data = np.load('../results/mfcc_features.npz')
X_train, y_train = data['X_train'], data['y_train']
X_test,  y_test  = data['X_test'],  data['y_test']

print(f'Train: {X_train.shape}  Test: {X_test.shape}')
print(f'Feature dim: {X_train.shape[1]}')

# Class distribution
unique, counts = np.unique(y_train, return_counts=True)
for u, c in zip(unique, counts):
    print(f'  class {EMOTION_NAMES[u]}: {c} samples')

## 2. Cross-Validation on Training Set

In [ ]:
classifiers = {
    'SVM (RBF)':   Pipeline([('scaler', StandardScaler()), ('clf', SVC(kernel='rbf', C=10, gamma='scale', class_weight='balanced'))]),
    'SVM (Linear)':Pipeline([('scaler', StandardScaler()), ('clf', SVC(kernel='linear', C=1, class_weight='balanced'))]),
    'Random Forest':Pipeline([('scaler', StandardScaler()), ('clf', RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42))]),
    'k-NN (k=5)':  Pipeline([('scaler', StandardScaler()), ('clf', KNeighborsClassifier(n_neighbors=5))]),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = {}

for name, clf in classifiers.items():
    scores = cross_val_score(clf, X_train, y_train, cv=cv, scoring='f1_macro', n_jobs=-1)
    cv_results[name] = scores
    print(f'{name:20s}  F1-macro: {scores.mean():.4f} ± {scores.std():.4f}')

## 3. Cross-Validation Results Plot

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
names = list(cv_results.keys())
means = [cv_results[n].mean() for n in names]
stds  = [cv_results[n].std()  for n in names]
colors = sns.color_palette('Set2', len(names))
bars = ax.bar(names, means, yerr=stds, capsize=5, color=colors, edgecolor='black', linewidth=0.5)
ax.set_ylim(0, 1)
ax.set_ylabel('F1 Macro (5-fold CV)')
ax.set_title('Baseline Classifier Comparison (Cross-Validation)')
for bar, m in zip(bars, means):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{m:.3f}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.savefig('../results/baseline_cv.png', dpi=150)
plt.show()

## 4. Final Evaluation on Test Set (Best Model)

In [ ]:
# Train best model on full training set
best_name = max(cv_results, key=lambda n: cv_results[n].mean())
print(f'Best baseline: {best_name}')

best_clf = classifiers[best_name]
best_clf.fit(X_train, y_train)
y_pred = best_clf.predict(X_test)

acc  = accuracy_score(y_test, y_pred)
f1m  = f1_score(y_test, y_pred, average='macro', zero_division=0)
f1w  = f1_score(y_test, y_pred, average='weighted', zero_division=0)
print(f'Test Accuracy   : {acc:.4f}')
print(f'Test F1 macro   : {f1m:.4f}')
print(f'Test F1 weighted: {f1w:.4f}')
print()
print(classification_report(y_test, y_pred, target_names=EMOTION_NAMES, zero_division=0))

## 5. Confusion Matrix

In [ ]:
from utils import plot_confusion_matrix
fig = plot_confusion_matrix(y_test, y_pred,
                             title=f'Confusion Matrix — {best_name}',
                             save_path='../results/cm_baseline.png')
plt.show()